In [ ]:
pip install streamlit langchain langchain-community langchain-ollama faiss-cpu pypdf

In [10]:
pip install langchain langchain-ollama

INFO: pip is looking at multiple versions of langchain-ollama to determine which version is compatible with other requirements. This could take a while.

   -------------------- ------------------- 1/2 [langchain-ollama]
   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\P1 Gen2\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [7]:
pip install langchain_community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\P1 Gen2\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [11]:
from langchain_ollama import ChatOllama

In [14]:
chat = ChatOllama(model = 'llama3.2:latest')

In [17]:
result = chat.invoke('What Claude Model')
print(result)

content="The Claude model, also known as the Claude Shannon model or Claude Shannon's mathematical model, is a mathematical model developed by Claude Shannon, an American mathematician and electrical engineer, in the 1940s. It is considered one of the most influential models in the history of information theory.\n\nThe Claude model is a simple, yet powerful, model that describes the fundamental principles of information transmission and processing. The model is based on the idea that information is represented by discrete symbols, such as bits, and that these symbols are transmitted through a channel, such as a wire or a radio wave.\n\nThe Claude model consists of three main components:\n\n1. **Source**: The source is the entity that generates the information to be transmitted, such as a person typing on a keyboard.\n2. **Channel**: The channel is the medium through which the information is transmitted, such as a wire or a radio wave.\n3. **Receiver**: The receiver is the entity that r

In [21]:
for m in chat.stream("What is LLM"):
    print(m.content, end="", flush=True)

LLM stands for Large Language Model. It's a type of artificial intelligence (AI) model that's specifically designed to process and understand human language. LLMs are trained on vast amounts of text data, which enables them to learn patterns, relationships, and structures of language.

Large Language Models are typically trained using a technique called masked language modeling, where a portion of the input text is randomly masked or removed, and the model is asked to predict the missing words. This process helps the model learn the context and relationships between words in language.

LLMs have several key characteristics that make them useful for a wide range of applications:

1. **Natural Language Understanding (NLU)**: LLMs can understand the meaning of text, including grammar, syntax, and semantics.
2. **Language Generation**: LLMs can generate human-like text based on the input they receive.
3. **Text Classification**: LLMs can classify text into categories, such as sentiment ana

### RAG Chatbot

In [22]:
from langchain_community.document_loaders import PyPDFLoader

In [50]:
loader = PyPDFLoader("pdfs/Aiesl Employees service regulation.pdf")
documents = loader.load()

context = "\n\n".join(doc.page_content for doc in documents)


In [25]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables = ["context", "question"],
    template = """
    You are an intelligent assistant trained to understand and provide information based
    on the context provided.

    Context: {context}
    Question: {question}

    Your responses should be directly address the question using the information from the context, while being clear and concise.
    """
)

In [35]:
formatted_prompt = prompt.format(
    context = context,
    question = "Can the Directorate General of Civil Aviation require additional medical checks?"
)

In [36]:
result = chat.invoke(formatted_prompt)

In [37]:
print(result)

content="There is no relevant information in the provided context about the Directorate General of Civil Aviation. The context appears to be a legal or regulatory document related to employment, discipline, and procedures for handling employee misconduct. \n\nHowever, I can attempt to provide a general answer based on typical practices in similar contexts.\n\nThe Directorate General of Civil Aviation (DGCA) is a regulatory body responsible for ensuring the safety and security of civil aviation in India. If the DGCA were to require additional medical checks, it would likely be related to the safety and well-being of employees in the aviation industry, such as pilots or air traffic controllers.\n\nIn the context of employment and disciplinary procedures, the DGCA would not typically be responsible for handling internal disciplinary issues within its own organization. However, if an employee working for the DGCA were to be involved in misconduct or required to undergo medical checks, the 

In [38]:
for m in chat.stream(formatted_prompt):
    print(m.content, end="", flush=True)

There is no information in the provided context about the Directorate General of Civil Aviation or medical checks. The context appears to be related to a company's internal policies and procedures for handling disciplinary actions, rather than a government agency or regulatory body.

### Directory Based Chatbot

In [57]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_core.prompts import PromptTemplate

loader = DirectoryLoader(
    path = "pdfs",
    glob = "*.pdf",
    loader_cls = PyPDFLoader
)


documents_ = loader.load()


context = "\n\n".join(doc.page_content for doc in documents_)



prompt = PromptTemplate(
    input_variables = ["context", "question"],
    template = """
    You are an intelligent assistant trained to understand and provide information based
    on the context provided.

    Context: {context}
    Question: {question}

    Your responses should directly address the question using information from the context, while being clear and concise.
    """
)


formatted_prompt = prompt.format(
    context = context,
    question = "Who is N. Chandrasekaran, Chairman, Tata Sons and Air India"
)

for m in chat.stream(formatted_prompt):
    print(m.content, end="", flush=True)


N. Chandrasekaran is the Chairman of Tata Sons, which is the holding company of the Tata Group, a multinational conglomerate with diverse businesses. As part of the Tata Group, Air India is also owned and operated by Tata Sons.

## NEW RAG Chatbot

In [58]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings

from langchain_ollama import ChatOllama

In [ ]:
UPLOAD_DIR = "uploads"
VECTOR_DIR = "vector_db"

os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(VECTOR_DIR, exist_ok=True)

print("Folders created")

In [ ]:
os.listdir(UPLOAD_DIR)

In [ ]:
documents = []

for file in os.listdir(UPLOAD_DIR):

    if file.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join(UPLOAD_DIR, file))
        docs = loader.load()

        documents.extend(docs)

print("Total pages loaded:", len(documents))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

### Create the Vector Database
##### We use FAISS, which is very fast.
vector_db/
    index.faiss
    index.pkl

In [ ]:
vectordb = FAISS.from_documents(chunks, embeddings)

vectordb.save_local(VECTOR_DIR)

print("Vector database created")

#### Step 9 — Load the Vector Database
Instead of recreating embeddings every time, we load the saved vector database.

In [ ]:
vectordb = FAISS.load_local(
    VECTOR_DIR,
    embeddings,
    allow_dangerous_deserialization=True
)

print("Vector database loaded")

#### Step 10 — Create the Retriever
The retriever finds relevant chunks of information from the vector database.
User Question -> Retriever -> Top 4 relevant chunks

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k":4})

####  Step 11 — Load the AI Model
Now we load the Large Language Model (LLM) that will generate answers.

We are using Llama 3.2 running locally with Ollama.

In [ ]:
llm = ChatOllama(model="llama3.2:latest")

#### Step 12 — Ask a Question
User Question -> Retriever finds relevant chunks -> Chunks sent to LLM -> LLM generates answer

In [ ]:
question = "Who is N. Chandrasekaran?"

docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
Use the context below to answer the question.

Context: {context}

Question: {question}
"""

response = llm.invoke(prompt)

print(response.content)

#### Step 13 — Show Source Citations
Users should know which document the answer came from.
- Sources used:
  - uploads/airline_policy.pdf

In [ ]:
sources = set()

for doc in docs:
    sources.add(doc.metadata["source"])

print("Sources used:")
for s in sources:
    print("-", s)

PDF Documents -> Text Extraction -> Chunking -> Embeddings -> Vector Database (FAISS) -> Retriever -> Relevant Context -> LLM (Llama 3.2) -> Answer + Sources